<a href="https://colab.research.google.com/github/adanbintewaqar/decode-labs-project-1/blob/main/Project_1_Advanced_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# 1. LOAD THE DATASET
file_path = 'Dataset for Data Analytics - Sheet1.csv'
df = pd.read_csv(file_path)

print(f"Original Dataset Shape: {df.shape}\n")

# ---------------------------------------------------------
# STAGE 1: INPUT FIDELITY & DATA CLEANING
# ---------------------------------------------------------
# A. Missing Value Imputation
# CouponCode has 25.75% missingness (>20% threshold) -> Impute with 'NO_COUPON'
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

# B. Outlier Neutralization (IQR Method + Winsorization/Clipping)
numeric_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap extreme values at lower/upper statistical boundaries
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("✅ Stage 1 Complete: Missing values imputed and outliers capped.")

# ---------------------------------------------------------
# STAGE 2: FEATURE ENGINEERING & COMPUTATION
# ---------------------------------------------------------
# Feature 1: Has_Coupon (Binary Indicator)
df['Has_Coupon'] = np.where(df['CouponCode'] == 'NO_COUPON', 0, 1)

# Feature 2: Is_Weekend (Extracted from Date)
df['Date'] = pd.to_datetime(df['Date'])
df['Order_Month'] = df['Date'].dt.month
df['Is_Weekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# Feature 3: AvgPricePerCartItem
df['AvgPricePerCartItem'] = df['TotalPrice'] / df['ItemsInCart']

# Feature 4: QuantityToCartRatio
df['QuantityToCartRatio'] = df['Quantity'] / df['ItemsInCart']

# Convert Categorical Columns to Coordinate Space (One-Hot Encoding)
categorical_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("✅ Stage 2 Complete: Engineered 4 predictive features & one-hot encoded categories.")

# ---------------------------------------------------------
# STAGE 3: RUNTIME DATA VALIDATION & EXPORT
# ---------------------------------------------------------
# Verify data contracts
assert df_encoded.isnull().sum().sum() == 0, "Validation Error: Null values found!"
assert (df_encoded['TotalPrice'] >= 0).all(), "Validation Error: Negative prices found!"

# Save processed output
output_file = 'Cleaned_Data_Analytics_Dataset.csv'
df_encoded.to_csv(output_file, index=False)

print(f"✅ Stage 3 Complete: Runtime checks passed. Clean dataset exported as '{output_file}'.\n")
print("--- PREVIEW OF CLEANED DATASET ---")
display(df_encoded.head())

Original Dataset Shape: (1200, 14)

✅ Stage 1 Complete: Missing values imputed and outliers capped.
✅ Stage 2 Complete: Engineered 4 predictive features & one-hot encoded categories.
✅ Stage 3 Complete: Runtime checks passed. Clean dataset exported as 'Cleaned_Data_Analytics_Dataset.csv'.

--- PREVIEW OF CLEANED DATASET ---


,OrderID,Date,CustomerID,Quantity,UnitPrice,ShippingAddress,TrackingNumber,ItemsInCart,CouponCode,TotalPrice,...,PaymentMethod_Gift Card,PaymentMethod_Online,OrderStatus_Delivered,OrderStatus_Pending,OrderStatus_Returned,OrderStatus_Shipped,ReferralSource_Facebook,ReferralSource_Google,ReferralSource_Instagram,ReferralSource_Referral
0,ORD200000,2023-01-04,C72649,5,570.62,928 Main St,TRK37947903,7,SAVE10,2853.10,...,False,False,False,False,False,True,False,False,True,False
1,ORD200001,2024-08-23,C75739,2,151.35,823 Main St,TRK91186779,3,SAVE10,302.70,...,False,True,False,False,False,True,False,False,False,True
2,ORD200002,2024-02-27,C81728,5,550.68,512 Main St,TRK42903982,8,FREESHIP,2753.40,...,False,False,False,False,False,False,False,False,False,False
3,ORD200003,2023-10-15,C33540,1,273.19,275 Main St,TRK62788070,5,SAVE10,273.19,...,False,False,False,False,True,False,True,False,False,False
4,ORD200004,2025-05-08,C81840,4,626.01,668 Main St,TRK29241424,8,SAVE10,2504.04,...,False,True,True,False,False,False,False,False,False,False
